# INTELIGÊNCIA ARTIFICIAL
## LABORATÓRIO # 08 — RAG com ChromaDB e Guardrails
**Do RAG Manual ao Banco Vetorial Persistente · Proteções de Entrada, Grounding e Tratamento Fora de Escopo**

---

### Objetivos do Laboratório
Evoluir a arquitetura do Laboratório 07 implementando:
1. **Banco Vetorial Persistente (ChromaDB):** Substituir a busca manual em memória por persistência em disco com `chromadb.PersistentClient`.
2. **Guardrail de Entrada:** Barrar perguntas vazias, caracteres inválidos ou consultas curtas antes de consumir requisições ao LLM.
3. **Guardrail de Grounding Numérico:** Estabelecer um limiar (*threshold*) de distância vetorial para detectar matematicamente quando o documento não possui a resposta requerida.
4. **Agente RAG Protegido:** Pipeline orquestrada de ponta a ponta resistente a alucinações e com máxima confiabilidade.


### 1. Preparação das Bibliotecas e Dependências
Execute caso necessário:
```bash
pip install chromadb groq pdfplumber python-dotenv sentence-transformers
```


In [ ]:
import os
import re
import pdfplumber
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

load_dotenv()
print(f"Versão do ChromaDB instalada: {chromadb.__version__}")

---
### 2. Configuração do ChromaDB Persistente e Coleção
Configuramos uma instância do ChromaDB que salva as informações em disco na pasta `./chroma_curso`.

In [ ]:
# 1. Criação do cliente persistente em disco
diretorio_chroma = "./chroma_curso"
chroma_client = chromadb.PersistentClient(path=diretorio_chroma)

# 2. Configuração da função de embedding (utilizando sentence-transformers)
try:
    emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="paraphrase-multilingual-MiniLM-L12-v2"
    )
    print("Função de embedding local configurada!")
except Exception as e:
    print(f"Aviso: Usando função padrão de embeddings do Chroma: {e}")
    emb_fn = embedding_functions.DefaultEmbeddingFunction()

# 3. Criação ou recuperação da coleção
colecao = chroma_client.get_or_create_collection(
    name="introducao_ml",
    embedding_function=emb_fn
)

print(f"Coleção 'introducao_ml' carregada. Documentos indexados atualmente: {colecao.count()}")

---
### 3. Indexação do Documento PDF no Banco Vetorial
Garantimos a inserção condicional (`if colecao.count() == 0`) para evitar duplicatas em execuções subsequentes.

In [ ]:
def ler_pdf(caminho):
    texto = ""
    with pdfplumber.open(caminho) as pdf:
        for p in pdf.pages:
            t = p.extract_text()
            if t: texto += t + " "
    return texto.strip()

arquivo_pdf = "introducaoml.pdf"
if os.path.exists(arquivo_pdf):
    texto_doc = ler_pdf(arquivo_pdf)
else:
    texto_doc = "Machine learning é um campo da IA que treina algoritmos a partir de dados. Existem três tipos principais: supervisionado, não supervisionado e por reforço."

# Divisão simples em chunks coerentes
chunks = [c.strip() for c in re.split(r'(?<=[.!?])\s+', texto_doc) if len(c.strip()) > 35]

# Adição persistente ao banco caso a base esteja vazia
if colecao.count() == 0:
    colecao.add(
        documents=chunks,
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        metadatas=[{"origem": arquivo_pdf, "posicao": i} for i in range(len(chunks))]
    )
    print(f"Indexação concluída! {len(chunks)} chunks inseridos no ChromaDB.")
else:
    print(f"A base já possui {colecao.count()} chunks. Reindexação não necessária.")

---
### 4. Busca Vetorial com `col.query()`
A busca vetorial é simplificada para apenas uma linha de código, retornando documentos e distâncias euclidianas/cosseno.

In [ ]:
def recuperar(pergunta, k=3):
    resultados = colecao.query(
        query_texts=[pergunta],
        n_results=k
    )
    documentos = resultados["documents"][0]
    distancias = resultados["distances"][0]  # Menor distância = Maior similaridade!
    return documentos, distancias

# Teste da recuperação
docs_teste, dists_teste = recuperar("O que é aprendizado de máquina?", k=2)
print("=== TESTE DE RECUPERAÇÃO ===")
for i, (d, s) in enumerate(zip(docs_teste, dists_teste), 1):
    print(f"Chunk {i} [Distância: {s:.4f}]: {d[:120]}...")

---
### 5. Implementação dos Guardrails (Proteções de Entrada e Grounding)

#### 5.1 Guardrail 1: Validação de Entrada
Filtra consultas vazias ou excessivamente curtas antes de onerar a infraestrutura ou o modelo.

In [ ]:
def guardrail_entrada(pergunta):
    pergunta_limpa = pergunta.strip()
    if len(pergunta_limpa) < 5:
        return False, "Pergunta inválida: a consulta é muito curta ou vazia."
    return True, None

#### 5.2 Guardrail 2: Validação de Grounding por Limiar Numérico
Caso a menor distância vetorial obtida seja superior ao limiar (`LIMIAR_DISTANCIA`), concluímos matematicamente que o banco vetorial não possui conteúdo suficiente para responder com segurança.

In [ ]:
LIMIAR_DISTANCIA = 1.25  # Limiar de corte de distância para modelo multilingual

def contexto_relevante(distancias, limiar=LIMIAR_DISTANCIA):
    # Se a lista de distâncias estiver vazia ou o melhor trecho for distante demais
    if not distancias or distancias[0] > limiar:
        return False
    return True

---
### 6. Pipeline Completa: RAG Protegido Ponta a Ponta

In [ ]:
from groq import Groq

groq_key = os.getenv("GROQ_API_KEY") or os.getenv("GROQ_API_KEY_2")
client_groq = Groq(api_key=groq_key) if groq_key else None
MODELO_LLM = "llama-3.1-8b-instant"

def responder_protegido(pergunta):
    # === GUARDRAIL 1: Entrada ===
    valido, msg_erro = guardrail_entrada(pergunta)
    if not valido:
        return f"🛡️ [GUARDRAIL ENTRADA ATIVADO]: {msg_erro}"
    
    # === RECUPERAÇÃO NO BANCO VETORIAL ===
    documentos, distancias = recuperar(pergunta, k=2)
    
    # === GUARDRAIL 2: Grounding Numérico ===
    if not contexto_relevante(distancias):
        return f"🛡️ [GUARDRAIL GROUNDING ATIVADO]: Informação não localizada na base (Menor distância = {distancias[0]:.4f} > limiar)."
    
    # === GERAÇÃO COM O LLM ===
    contexto = "\n".join(documentos)
    prompt = f"""Responda a pergunta utilizando APENAS o contexto abaixo fornecido.
Se a informação não for conclusiva pelo contexto, diga que não sabe.

CONTEXTO:
{contexto}

PERGUNTA:
{pergunta}
"""
    if client_groq is not None:
        try:
            resp = client_groq.chat.completions.create(
                model=MODELO_LLM,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2
            )
            return resp.choices[0].message.content
        except Exception as e:
            return f"[Erro na chamada Groq: {e}]"
    else:
        # Fallback offline demonstrativo
        return f"[Simulação com sucesso]: Com base no documento, 'Aprendizado de Máquina' refere-se ao campo da IA focado no aprendizado por meio de dados." 

---
### 7. Bateria de Testes com as Proteções em Ação

In [ ]:
casos_de_teste = [
    ("O que é aprendizado de máquina?", "Consulta normal no domínio do documento"),
    ("oi", "Entrada inválida/curta (deve disparar Guardrail 1)"),
    ("Qual a receita de lasanha à bolonhesa?", "Fora de escopo (deve disparar Guardrail 2)")
]

for pergunta, descricao in casos_de_teste:
    print("=" * 70)
    print(f"Cenário: {descricao}")
    print(f"Pergunta: '{pergunta}'")
    resposta = responder_protegido(pergunta)
    print(f"Saída:\n{resposta}\n")

---
### 8. Quadro-Resumo da Arquitetura Segura

| Componente | Papel Arquitetural | Benefício Prático |
| :--- | :--- | :--- |
| **ChromaDB** | Banco vetorial indexado em disco | Consultas rápidas, persistência automática e sem duplicatas |
| **Guardrail de Entrada** | Validação sintática preliminar | Reduz latência e economiza custos de requisição à API do LLM |
| **Guardrail de Grounding** | Verificação vetorial da distância | Elimina alucinações cortando perguntas fora de escopo |
| **Prompt com Grounding** | Instrução de contenção ao modelo | Garante respostas aderentes estritamente às evidências |

Parabéns! Você concluiu toda a jornada prática dos Laboratórios de Inteligência Artificial:
- **Fundamentos & Pandas** (Labs 00 e 01)
- **Engenharia de Dados & EDA** (Labs 02 e 03)
- **Modelagem e Avaliação de Machine Learning** (Lab 04)
- **Processamento de Linguagem Natural Clássico** (Lab 05)
- **Embeddings & Busca Semântica** (Lab 06)
- **RAG Completo com LLMs** (Lab 07)
- **Bancos Vetoriais & Guardrails de Produção** (Lab 08)
